In [4]:
# ============================================================================
# SME BUSINESS LENDING – FULL AGENT SYSTEM IN ONE CELL
# ============================================================================
# This cell gives you:
#   • A generic Agent / AgentTeam framework with a mock LLM client.
#   • A library of domain-specific SME business lending agents.
#   • A mapping from product-development phases → recommended agents.
#
# You can:
#   • Inspect the framework: describe_phase("2.2")
#   • Instantiate phase-specific agents: build_phase_agents("2.2")
#   • Wrap them in an AgentTeam and run workflows.
# ============================================================================

# -----------------------------
# Imports & basic plumbing
# -----------------------------
import os
from enum import Enum
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Callable, Tuple

# ============================================================================
# 1. EXECUTION MODES & MOCK LLM CLIENT
# ============================================================================

class ExecutionMode(str, Enum):
    """
    Execution mode controls HOW each Agent calls an LLM.

    In this minimal version we support:
      • MOCK   – deterministic, fast fake responses (no API key needed).
      • ONLINE – placeholder if you want to wire in a real LLM later.
    """
    MOCK = "mock"
    ONLINE = "online"


DEFAULT_MODE: ExecutionMode = ExecutionMode.MOCK


class MockClient:
    """
    Very small fake LLM client.

    It ignores the actual prompt, but:
      • Echoes back the user's input.
      • Adds a short role-specific prefix so you can see which agent replied.

    This keeps the notebook self-contained and runnable with no API keys.
    """

    def generate(
        self,
        role: str,
        system_prompt: str,
        user_input: str,
        history: List[Dict[str, str]],
    ) -> str:
        role_descriptions = {
            "researcher": "Quick research summary",
            "analyst": "Analytical breakdown",
            "writer": "Structured explanation",
            "critic": "Critical review",
            "coordinator": "Orchestration / plan",
            "executor": "Action-oriented response",
        }
        descriptor = role_descriptions.get(role, "Generic response")
        return (
            f"[MOCK: {descriptor} from a '{role}' agent]\n\n"
            f"System context (truncated): {system_prompt[:160]}...\n\n"
            f"User request:\n{user_input}\n\n"
            f"(In a real setup, this is where the model's answer would appear.)"
        )


class OnlineClient:
    """
    Placeholder for a real LLM client (OpenAI, local model, etc.).

    To keep this notebook self-contained and runnable everywhere,
    this client just raises a NotImplementedError.

    If you want to hook a real LLM:
      • Replace generate(...) with a call to your chat/completions API.
    """

    def generate(
        self,
        role: str,
        system_prompt: str,
        user_input: str,
        history: List[Dict[str, str]],
    ) -> str:
        raise NotImplementedError(
            "ONLINE mode is a placeholder. "
            "Wire this to your preferred LLM client if needed."
        )


# ============================================================================
# 2. CORE AGENT AND TEAM ABSTRACTIONS
# ============================================================================

class AgentRole(str, Enum):
    """
    Broad role category used for:
      • Mock responses (different flavour per role).
      • Any routing / orchestration logic you may want later.
    """
    RESEARCHER = "researcher"
    ANALYST = "analyst"
    WRITER = "writer"
    CRITIC = "critic"
    COORDINATOR = "coordinator"
    EXECUTOR = "executor"


@dataclass
class AgentMessage:
    """
    Lightweight container for an agent's reply.
    """
    agent_name: str
    role: AgentRole
    content: str
    thinking: Optional[str] = None
    metadata: Optional[Dict[str, Any]] = None


class Agent:
    """
    A single LLM-based agent with:

      • name         – short identifier (e.g. "CreditAssessment").
      • role         – AgentRole (e.g. ANALYST / RESEARCHER / COORDINATOR).
      • system_prompt– long description of its responsibility.
      • mode         – ExecutionMode (MOCK or ONLINE).
      • history      – simple conversation history (for context if you later
                       plug in a real LLM instead of MockClient).

    Use Agent.ask("some instruction") to interact with it.
    """

    def __init__(
        self,
        name: str,
        role: AgentRole,
        system_prompt: str,
        mode: ExecutionMode = DEFAULT_MODE,
    ):
        self.name = name
        self.role = role
        self.system_prompt = system_prompt
        self.mode = mode
        self.conversation_history: List[Dict[str, str]] = []
        self.client = self._init_client()

    # --------------------------
    # LLM client selection
    # --------------------------
    def _init_client(self):
        if self.mode == ExecutionMode.MOCK:
            return MockClient()
        elif self.mode == ExecutionMode.ONLINE:
            return OnlineClient()
        else:
            raise ValueError(f"Unsupported execution mode: {self.mode}")

    # --------------------------
    # Single-agent interaction
    # --------------------------
    def ask(
        self,
        user_input: str,
        metadata: Optional[Dict[str, Any]] = None,
    ) -> AgentMessage:
        """
        Send a prompt to this agent and get a structured AgentMessage back.
        """
        # Store user message in history (useful if you later wire up a real LLM)
        self.conversation_history.append({"role": "user", "content": user_input})

        response_text = self.client.generate(
            role=self.role.value,
            system_prompt=self.system_prompt,
            user_input=user_input,
            history=self.conversation_history,
        )

        self.conversation_history.append(
            {"role": "assistant", "content": response_text}
        )

        return AgentMessage(
            agent_name=self.name,
            role=self.role,
            content=response_text,
            thinking=None,
            metadata=metadata or {},
        )


class AgentTeam:
    """
    Simple orchestrator that groups multiple Agents and lets you:

      • Add agents with add_agent(...)
      • See available agents with list_agents()
      • Broadcast the same prompt to all via broadcast(...)
      • Run simple sequential workflows using run_sequential_workflow(...)
    """

    def __init__(self, name: str, mode: ExecutionMode = DEFAULT_MODE):
        self.name = name
        self.mode = mode
        self.agents: Dict[str, Agent] = {}

    # --------------------------
    # Team management helpers
    # --------------------------
    def add_agent(self, agent: Agent):
        self.agents[agent.name] = agent

    def get_agent(self, name: str) -> Agent:
        return self.agents[name]

    def list_agents(self) -> List[str]:
        return list(self.agents.keys())

    # --------------------------
    # Multi-agent workflows
    # --------------------------
    def broadcast(self, prompt: str) -> Dict[str, AgentMessage]:
        """
        Send the same prompt to all agents in the team.
        """
        results: Dict[str, AgentMessage] = {}
        for name, agent in self.agents.items():
            results[name] = agent.ask(prompt)
        return results

    def run_sequential_workflow(
        self,
        steps: List[Tuple[str, str]],
        verbose: bool = True,
    ) -> List[AgentMessage]:
        """
        Run a simple sequential workflow, where each step is:

            ("AgentName", "instruction for this step")

        The results are returned as a list of AgentMessage objects.
        """
        results: List[AgentMessage] = []
        for agent_name, instruction in steps:
            if agent_name not in self.agents:
                raise KeyError(
                    f"Agent '{agent_name}' not in team '{self.name}'. "
                    f"Available: {self.list_agents()}"
                )
            agent = self.agents[agent_name]
            if verbose:
                print(f"\n--- [{self.name}] {agent.name} ({agent.role.value}) ---")
                print("Instruction:")
                print(instruction)
                print()
            msg = agent.ask(instruction)
            results.append(msg)
            if verbose:
                print("Response (truncated):")
                print(msg.content[:400] + ("..." if len(msg.content) > 400 else ""))
                print()
        return results


# ============================================================================
# 3. SME BUSINESS LENDING – DOMAIN CONTEXT AND FACTORY
# ============================================================================

BASE_SME_CONTEXT = """
You are part of a multi-agent AI system for designing, launching and operating
an SME business lending product in Europe (micro, small and medium enterprises).

You must:
- Reflect good practice in banking, credit risk and regulatory expectations
  at a conceptual level (note, you may NOT provide legal or regulatory advice).
- Be explicit about assumptions and data gaps.
- Clearly identify risks, dependencies, and where human oversight is required.
- Write in structured form using headings and bullet points where useful.
""".strip()

def make_domain_agent(
    name: str,
    mission: str,
    key_tasks: str,
    *,
    mode: ExecutionMode = DEFAULT_MODE,
    agent_role: AgentRole = AgentRole.ANALYST,
) -> Agent:
    """
    Generic factory for SME product-development agents.

    Arguments:
      • name       – agent name (also used as Agent.name).
      • mission    – 1–2 line summary of the agent's purpose.
      • key_tasks  – bullet list string with core responsibilities.
      • mode       – ExecutionMode.
      • agent_role – broad role category for orchestration.
    """
    system_prompt = f"""
{BASE_SME_CONTEXT}

You are the **{name}** in this system.

Primary mission:
- {mission}

Key responsibilities:
{key_tasks}

Working style:
- Propose concrete artefacts (tables, bullet lists, decision rules, checklists).
- Call out data gaps, assumptions, and validation steps.
- Make your reasoning explicit enough for other agents and humans to reuse.
""".strip()

    return Agent(
        name=name,
        role=agent_role,
        system_prompt=system_prompt,
        mode=mode,
    )


# ============================================================================
# 4. SME AGENT FACTORIES (ALIGNED TO PRODUCT-DEV FRAMEWORK)
# ============================================================================

# -------------------------------
# Phase 1 – Strategy & Discovery
# -------------------------------

def make_data_collection_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Data Collection Agent",
        "collect structured data about SMEs, competitors and regulation.",
        "- Identify and summarise official statistics, market studies and trade bodies.\n"
        "- Capture competitor product features, pricing and distribution channels.\n"
        "- Log data sources, coverage, quality issues and update frequency.",
        mode=mode,
        agent_role=AgentRole.RESEARCHER,
    )


def make_segmentation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Segmentation Agent",
        "define SME market segments and size the opportunity.",
        "- Segment by size, sector, age, geography and digital maturity.\n"
        "- Estimate TAM / SAM / SOM by segment based on available data.\n"
        "- Highlight underserved niches and unattractive segments.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_trend_analysis_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Trend Analysis Agent",
        "analyse historical and emerging trends in SME lending.",
        "- Track growth rates, default rates and pricing trends by segment.\n"
        "- Identify macro and structural shifts (e.g. fintech penetration).\n"
        "- Summarise opportunities and threats over a 3–5 year horizon.",
        mode=mode,
        agent_role=AgentRole.RESEARCHER,
    )


def make_survey_analysis_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Survey Analysis Agent",
        "convert SME interviews and surveys into prioritised customer needs.",
        "- Cluster qualitative feedback into themes and pain points.\n"
        "- Map pains to product features, service elements and SLAs.\n"
        "- Propose a short list of top problems to solve first.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_financial_modelling_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Financial Modelling Agent",
        "build high-level financial projections for the SME lending product.",
        "- Define volume, margin, loss and cost assumptions by segment.\n"
        "- Construct base / upside / downside scenarios and sensitivities.\n"
        "- Estimate breakeven, ROA and ROE over a 3–5 year horizon.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_competitor_pricing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Competitor Pricing Agent",
        "summarise competitor pricing structures and positioning.",
        "- Compare headline APRs, fees and non-price features.\n"
        "- Identify aggressive vs conservative players and their likely strategy.\n"
        "- Suggest where to position initial pricing bands.",
        mode=mode,
        agent_role=AgentRole.RESEARCHER,
    )


def make_cost_analytics_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Cost Analytics Agent",
        "estimate and explain cost-to-serve for the product.",
        "- Break down acquisition, underwriting, servicing and collections costs.\n"
        "- Express unit economics per application and per loan.\n"
        "- Identify levers to reduce cost while maintaining control quality.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_capital_modelling_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Capital Modelling Agent",
        "estimate capital consumption for the SME portfolio at a high level.",
        "- Apply simple risk-weight or capital charge assumptions by segment.\n"
        "- Project capital usage under different growth and mix scenarios.\n"
        "- Highlight capital constraints and mitigation options conceptually.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_regulatory_classification_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Regulatory Classification Agent",
        "clarify regulatory perimeter and relevant rulebooks for the product.",
        "- Distinguish business vs consumer lending dimensions.\n"
        "- Identify which regimes, permissions and codes of conduct seem relevant.\n"
        "- Summarise open questions and assumptions for legal/compliance to confirm.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_capital_requirement_calculator_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Capital Requirement Calculator Agent",
        "translate portfolio assumptions into indicative capital requirements.",
        "- Combine portfolio mix, risk weights and size into capital estimates.\n"
        "- Present simple tables showing capital usage by scenario.\n"
        "- Explain sensitivities and limitations of the approach.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_compliance_checklist_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Compliance Checklist Agent",
        "turn regulatory expectations into actionable task lists.",
        "- Break down each relevant regulation into concrete actions and artefacts.\n"
        "- Tag owners, dependencies and suggested timelines.\n"
        "- Highlight critical-path items for launch.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_regulatory_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Regulatory Monitoring Agent",
        "track regulatory developments that might affect the product.",
        "- Outline processes for monitoring new guidance and consultations.\n"
        "- Propose impact assessment steps for changes.\n"
        "- Suggest governance for updating policies and product design.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


# ------------------------------------------
# Phase 2 – Governance & Credit Risk Setup
# ------------------------------------------

def make_policy_compliance_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Policy Compliance Agent",
        "check lending decisions and processes against credit policy and risk appetite.",
        "- Map policy statements to specific decision rules and thresholds.\n"
        "- Identify potential breaches or grey areas conceptually.\n"
        "- Suggest additional MI required to evidence adherence.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_governance_workflow_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Governance Workflow Agent",
        "design governance structures and escalation paths for product and credit decisions.",
        "- Define committee structure, membership and decision rights.\n"
        "- Map which items go to which forum and when.\n"
        "- Propose documentation and audit trail expectations.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_risk_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Risk Monitoring Agent",
        "monitor overall credit risk evolution and limit usage conceptually.",
        "- Define key risk indicators and portfolio limits.\n"
        "- Propose how to track them over time.\n"
        "- Suggest actions if limits are approached or breached.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_data_quality_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Data Quality Agent",
        "define and monitor data-quality requirements for SME lending data.",
        "- Specify critical fields and validation rules.\n"
        "- Propose monitoring dashboards and acceptance thresholds.\n"
        "- Outline remediation workflows when issues are found.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_financial_analysis_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Financial Analysis Agent",
        "analyse SME financial statements and bank data.",
        "- Calculate key ratios (profitability, leverage, coverage, liquidity).\n"
        "- Identify trends, anomalies and red flags.\n"
        "- Summarise strengths, weaknesses and data gaps.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_credit_assessment_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Credit Assessment Agent",
        "synthesise financial, qualitative and sector information into a credit view.",
        "- Draft structured credit assessments with pros/cons.\n"
        "- Identify risk drivers and mitigants (collateral, guarantees, covenants).\n"
        "- Suggest approval/decline and conditions conceptually.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_credit_scoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Credit Scoring Agent",
        "design and interpret scorecard-style credit scoring for SMEs.",
        "- Propose input variables and how they influence the score.\n"
        "- Map scores to qualitative risk bands.\n"
        "- Explain limitations and monitoring needs for models.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_decisioning_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Decisioning Agent",
        "apply approval, refer and decline rules for SME lending.",
        "- Translate policy and risk appetite into decision flows.\n"
        "- Distinguish auto, manual and escalated paths.\n"
        "- Document rationale and conditions for each branch.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_pricing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Pricing Agent",
        "calculate risk-based APRs and fees for SME loans.",
        "- Combine cost of funds, expected loss, capital cost and margin.\n"
        "- Express results as grids by segment, risk band and tenor.\n"
        "- Check alignment with policy and competitiveness conceptually.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_ecl_calculation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "ECL Calculation Agent",
        "outline expected credit loss (ECL) methodology at a conceptual level.",
        "- Explain Stage 1/2/3 with PD, LGD and EAD inputs.\n"
        "- Provide example segment-level calculations.\n"
        "- Highlight assumptions, limitations and potential overlays.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_concentration_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Concentration Monitoring Agent",
        "monitor concentrations by sector, geography and single name conceptually.",
        "- Define simple concentration metrics and limits.\n"
        "- Suggest ways to visualise and track them.\n"
        "- Outline management actions when concentrations build.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# -------------------------------------
# Phase 3 – Product & Pricing Design
# -------------------------------------

def make_product_configuration_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Product Configuration Agent",
        "maintain structured definitions of SME loan products and variants.",
        "- Specify parameters (tenor, amount, pricing, security, eligibility).\n"
        "- Distinguish core products vs variants by segment.\n"
        "- Provide machine-readable structures for technology teams.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_customer_journey_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Customer Journey Agent",
        "design the end-to-end SME customer journey.",
        "- Map steps from awareness to application, approval and servicing.\n"
        "- Identify friction points and improvement opportunities.\n"
        "- Propose copy, timing and channels for key interactions.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_information_provision_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Information Provision Agent",
        "assemble pre-contract information and disclosures.",
        "- Summarise key terms, pricing and rights clearly.\n"
        "- Propose key fact documents and indicative layouts.\n"
        "- Highlight how information supports informed decision-making.",
        mode=mode,
        agent_role=AgentRole.WRITER,
    )


def make_pricing_analytics_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Pricing Analytics Agent",
        "analyse pricing performance and recommend adjustments.",
        "- Assess profitability by segment and risk band.\n"
        "- Identify under- and over-priced niches conceptually.\n"
        "- Suggest targeted changes and expected directional impact.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# -------------------------------------
# Phase 4 – Technology & Architecture
# -------------------------------------

def make_data_integration_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Data Integration Agent",
        "design how external and internal data sources connect into the platform.",
        "- Identify key data sources (registries, bureaus, open banking, internal systems).\n"
        "- Propose integration patterns and error-handling strategies.\n"
        "- Suggest metadata and lineage documentation.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_application_processing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Application Processing Agent",
        "orchestrate the internal application workflow and statuses.",
        "- Define application states and transitions (e.g. draft → submitted → assessed → approved).\n"
        "- Map which systems/teams act at each step.\n"
        "- Suggest SLAs and basic monitoring metrics.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_underwriting_automation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Underwriting Automation Agent",
        "identify and design automation opportunities in underwriting.",
        "- Propose rules suitable for straight-through processing.\n"
        "- Define when to route to human underwriters.\n"
        "- Outline controls to avoid unintended bias or policy breaches.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_servicing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Servicing Agent",
        "design servicing workflows after drawdown.",
        "- Map lifecycle events (payment, arrears, restructuring) to actions.\n"
        "- Propose communication triggers and internal tasks.\n"
        "- Suggest simple data structures for balances and schedules.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_regulatory_reporting_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Regulatory Reporting Agent",
        "outline regulatory and internal reporting requirements for the SME portfolio.",
        "- List key reports, their granularity and frequency conceptually.\n"
        "- Map necessary data fields to these reports at a high level.\n"
        "- Suggest controls and reconciliations to support accuracy.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_compliance_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Compliance Agent",
        "coordinate compliance requirements across technology and process design.",
        "- Ensure key regulatory requirements are represented in functional designs.\n"
        "- Identify where additional controls or evidence are needed.\n"
        "- Flag open issues for compliance/legal review.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


# -------------------------------------
# Phase 5 – AML / KYC Framework
# -------------------------------------

def make_kyc_verification_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "KYC Verification Agent",
        "define KYC requirements for SME entities, directors and UBOs.",
        "- Outline data and documents required for identification.\n"
        "- Propose verification checks and risk-based tiering.\n"
        "- Suggest how to record decisions and evidence.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_sanctions_screening_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Sanctions Screening Agent",
        "design sanctions/PEP screening scenarios and escalation logic conceptually.",
        "- Define which names and attributes to screen.\n"
        "- Propose match thresholds and triage rules.\n"
        "- Outline documentation expected for decisions.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_adverse_news_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Adverse News Monitoring Agent",
        "monitor adverse news about SME customers conceptually.",
        "- Identify types of relevant adverse events.\n"
        "- Propose monitoring frequency and sources.\n"
        "- Suggest scoring and escalation approach.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_transaction_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Transaction Monitoring Agent",
        "outline transaction monitoring scenarios for SME accounts.",
        "- Describe behavioural patterns that may indicate suspicious activity.\n"
        "- Propose threshold-based and pattern-based rules conceptually.\n"
        "- Suggest documentation requirements for investigations.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_sar_preparation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "SAR Preparation Agent",
        "help structure suspicious activity report narratives conceptually.",
        "- Summarise factual patterns and indicators.\n"
        "- Suggest how to structure the narrative clearly.\n"
        "- Flag confidentiality and tipping-off considerations conceptually.",
        mode=mode,
        agent_role=AgentRole.WRITER,
    )


def make_aml_compliance_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "AML Compliance Agent",
        "coordinate the overall AML/CTF framework at a conceptual level.",
        "- Summarise key policy requirements across the lifecycle.\n"
        "- Map controls to processes and data sources.\n"
        "- Suggest key metrics and evidence for oversight.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


# -------------------------------------------
# Phase 6 – Lending Standards & Affordability
# -------------------------------------------

def make_affordability_assessment_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Affordability Assessment Agent",
        "outline how to assess borrower affordability beyond PD/credit risk.",
        "- Distinguish affordability from credit risk conceptually.\n"
        "- Propose metrics (e.g. DSCR, cash-flow coverage) by product type.\n"
        "- Suggest thresholds and escalation guidelines at a high level.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_complaints_routing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Complaints Routing Agent",
        "design complaint logging, routing and tracking flows.",
        "- Categorise complaint types and severities.\n"
        "- Propose routing rules and SLAs.\n"
        "- Suggest MI for root-cause analysis and redress.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_vulnerability_detection_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Vulnerability Detection Agent",
        "identify signals of potentially vulnerable SME owners or guarantors at a conceptual level.",
        "- Propose indicators of vulnerability based on behaviour and information.\n"
        "- Suggest additional support and communication adaptations.\n"
        "- Emphasise non-discriminatory, fair-treatment principles.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_forbearance_management_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Forbearance Management Agent",
        "design approaches for supporting distressed but viable borrowers.",
        "- Outline standard forbearance options and decision criteria.\n"
        "- Suggest monitoring of forborne exposures.\n"
        "- Highlight risks and governance expectations conceptually.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


# -----------------------------------------------
# Phase 7 – Product Governance & Marketing
# -----------------------------------------------

def make_product_governance_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Product Governance Agent",
        "maintain the product, target market and change log documentation.",
        "- Define positive and negative target markets.\n"
        "- Record material changes and required approvals.\n"
        "- Propose regular review cycles and metrics.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_marketing_compliance_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Marketing Compliance Agent",
        "review marketing and sales material for fairness and clarity at a conceptual level.",
        "- Check key statements for balance and clarity.\n"
        "- Ensure pricing examples and qualifications are used appropriately.\n"
        "- Suggest tone and presentation improvements.",
        mode=mode,
        agent_role=AgentRole.CRITIC,
    )


def make_incentive_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Incentive Monitoring Agent",
        "assess sales incentive structures conceptually for potential conflicts.",
        "- Map incentive schemes and relevant controls.\n"
        "- Identify where incentives could distort behaviour.\n"
        "- Suggest safer, outcome-aligned structures.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_target_market_monitor_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Target Market Monitor Agent",
        "check whether realised customers match the defined target market at a high level.",
        "- Propose ways to compare actual vs intended customer profiles.\n"
        "- Flag drifts and potential causes conceptually.\n"
        "- Suggest corrective actions (eligibility, distribution, comms).",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# -------------------------------------------------
# Phase 8 – Testing & Production Readiness
# -------------------------------------------------

def make_test_case_generation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Test Case Generation Agent",
        "generate structured test cases for SME lending flows.",
        "- Derive test cases from requirements and user journeys.\n"
        "- Cover positive, negative and edge cases.\n"
        "- Tag tests by risk and priority conceptually.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_regression_testing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Regression Testing Agent",
        "design regression test suites for future changes.",
        "- Group tests into suites based on functionality and risk.\n"
        "- Propose triggers to re-run suites (e.g. model change, vendor upgrade).\n"
        "- Highlight critical areas prone to regression.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_uat_coordination_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "UAT Coordination Agent",
        "coordinate user acceptance testing conceptually.",
        "- Define user personas and key scenarios.\n"
        "- Propose UAT entry/exit criteria.\n"
        "- Suggest tracking of issues and sign-off.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_compliance_testing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Compliance Testing Agent",
        "design tests for control effectiveness (AML, conduct, GDPR) conceptually.",
        "- Identify critical controls to test.\n"
        "- Outline test steps and expected evidence.\n"
        "- Suggest how to document residual risks.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_performance_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Performance Monitoring Agent",
        "specify dashboards and KPIs to monitor early performance.",
        "- Define key funnel, risk, operational and customer KPIs conceptually.\n"
        "- Suggest thresholds and alert rules.\n"
        "- Propose review cadences and owners.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_readiness_assessment_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Readiness Assessment Agent",
        "track production readiness against a cross-functional checklist.",
        "- Maintain a checklist covering tech, ops, risk, compliance and people.\n"
        "- Flag gaps, owners and dates.\n"
        "- Suggest go/no-go criteria at a conceptual level.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


# -------------------------------------------------
# Phase 9 – Launch & Initial Operations
# -------------------------------------------------

def make_application_routing_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Application Routing Agent",
        "design routing rules for incoming applications.",
        "- Segment applications by size, complexity, risk or channel.\n"
        "- Propose queues and assignment logic.\n"
        "- Suggest metrics for routing effectiveness.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_performance_dashboard_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Performance Dashboard Agent",
        "design operational and strategic dashboards for the live product.",
        "- Propose dashboard layouts and core metrics.\n"
        "- Distinguish leading vs lagging indicators.\n"
        "- Suggest review cadence and ownership.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_issue_escalation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Issue Escalation Agent",
        "triage operational incidents, complaints and risk alerts conceptually.",
        "- Categorise issues by severity and type.\n"
        "- Propose escalation paths and timelines.\n"
        "- Suggest recording for problem management.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_customer_communication_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Customer Communication Agent",
        "draft customer-facing communications for key lifecycle events.",
        "- Propose templates for approvals, declines and information requests.\n"
        "- Suggest tone, structure and clarity improvements.\n"
        "- Highlight where tailoring or translation may be needed.",
        mode=mode,
        agent_role=AgentRole.WRITER,
    )


def make_customer_feedback_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Customer Feedback Agent",
        "analyse customer feedback to identify early issues and opportunities.",
        "- Aggregate NPS, survey and complaint themes.\n"
        "- Prioritise issues by frequency and impact.\n"
        "- Link insights to concrete improvement ideas.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# -------------------------------------------------
# Phase 10 – Ongoing Monitoring & Optimisation
# -------------------------------------------------

def make_portfolio_analytics_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Portfolio Analytics Agent",
        "analyse portfolio composition and performance conceptually.",
        "- Slice exposures, losses and profit by segment.\n"
        "- Compare actual vs planned volumes and risk.\n"
        "- Surface emerging issues and attractive niches.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_default_management_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Default Management Agent",
        "outline workflows for loans that default.",
        "- Define default triggers and subsequent steps.\n"
        "- Coordinate with legal and collections conceptually.\n"
        "- Suggest data capture for recoveries and write-offs.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_model_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Model Monitoring Agent",
        "suggest how to monitor model performance and drift conceptually.",
        "- Compare predicted vs realised outcomes.\n"
        "- Track approval rates and score distributions.\n"
        "- Flag when recalibration or development might be needed.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_provisioning_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Provisioning Agent",
        "coordinate provisioning / ECL at portfolio level conceptually.",
        "- Aggregate ECL by segment and portfolio.\n"
        "- Explain key drivers of changes period-on-period.\n"
        "- Explore the directional impact of macro overlays conceptually.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_compliance_monitoring_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Compliance Monitoring Agent",
        "track core compliance metrics and reporting obligations conceptually.",
        "- List key compliance indicators and reports.\n"
        "- Suggest high-level dashboards.\n"
        "- Flag patterns that warrant deep-dive.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# -------------------------------------------------
# Phase 11 – Strategic Optimisation & Scaling
# -------------------------------------------------

def make_segment_analytics_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Segment Analytics Agent",
        "assess profitability and risk by segment conceptually.",
        "- Build simple segment-level P&Ls.\n"
        "- Identify high- and low-performing segments.\n"
        "- Suggest segment-level strategy (grow/hold/exit).",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_market_expansion_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Market Expansion Agent",
        "evaluate expansion into new geographies or product adjacencies conceptually.",
        "- Size new markets and assess competitiveness.\n"
        "- Highlight regulatory and operational challenges.\n"
        "- Suggest pilot approaches and success criteria.",
        mode=mode,
        agent_role=AgentRole.RESEARCHER,
    )


def make_portfolio_optimisation_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Portfolio Optimisation Agent",
        "explore how changes in mix, pricing or strategy affect portfolio outcomes conceptually.",
        "- Outline scenarios for rebalancing the portfolio.\n"
        "- Discuss directional impacts on risk and profitability.\n"
        "- Suggest potential constraints and trade-offs.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_automation_opportunity_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Automation Opportunity Agent",
        "identify high-value automation use cases across the lifecycle conceptually.",
        "- Map manual processes and volumes.\n"
        "- Estimate complexity vs benefit of automation options.\n"
        "- Prioritise a roadmap of candidate use cases.",
        mode=mode,
        agent_role=AgentRole.COORDINATOR,
    )


def make_risk_scenario_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Risk Scenario Agent",
        "design stress and scenario tests for the SME portfolio conceptually.",
        "- Describe plausible macro and sector stress scenarios.\n"
        "- Explain expected directional impact on losses and capital.\n"
        "- Suggest contingency planning themes.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


def make_customer_lifetime_value_agent(mode: ExecutionMode = DEFAULT_MODE) -> Agent:
    return make_domain_agent(
        "Customer Lifetime Value Agent",
        "outline methods to estimate CLV and link to strategy conceptually.",
        "- Define CLV inputs (margin, retention, discounting).\n"
        "- Compare CLV vs acquisition cost conceptually.\n"
        "- Suggest actions to improve lifetime value.",
        mode=mode,
        agent_role=AgentRole.ANALYST,
    )


# ============================================================================
# 5. AGENT REGISTRY & PHASE MAPPING
# ============================================================================

SME_AGENT_REGISTRY: Dict[str, Callable[[ExecutionMode], Agent]] = {
    # Phase 1
    "Data Collection Agent": make_data_collection_agent,
    "Segmentation Agent": make_segmentation_agent,
    "Trend Analysis Agent": make_trend_analysis_agent,
    "Survey Analysis Agent": make_survey_analysis_agent,
    "Financial Modelling Agent": make_financial_modelling_agent,
    "Competitor Pricing Agent": make_competitor_pricing_agent,
    "Cost Analytics Agent": make_cost_analytics_agent,
    "Capital Modelling Agent": make_capital_modelling_agent,
    "Regulatory Classification Agent": make_regulatory_classification_agent,
    "Capital Requirement Calculator Agent": make_capital_requirement_calculator_agent,
    "Compliance Checklist Agent": make_compliance_checklist_agent,
    "Regulatory Monitoring Agent": make_regulatory_monitoring_agent,

    # Phase 2
    "Policy Compliance Agent": make_policy_compliance_agent,
    "Governance Workflow Agent": make_governance_workflow_agent,
    "Risk Monitoring Agent": make_risk_monitoring_agent,
    "Data Quality Agent": make_data_quality_agent,
    "Financial Analysis Agent": make_financial_analysis_agent,
    "Credit Assessment Agent": make_credit_assessment_agent,
    "Credit Scoring Agent": make_credit_scoring_agent,
    "Decisioning Agent": make_decisioning_agent,
    "Pricing Agent": make_pricing_agent,
    "ECL Calculation Agent": make_ecl_calculation_agent,
    "Concentration Monitoring Agent": make_concentration_monitoring_agent,

    # Phase 3
    "Product Configuration Agent": make_product_configuration_agent,
    "Customer Journey Agent": make_customer_journey_agent,
    "Information Provision Agent": make_information_provision_agent,
    "Pricing Analytics Agent": make_pricing_analytics_agent,

    # Phase 4
    "Data Integration Agent": make_data_integration_agent,
    "Application Processing Agent": make_application_processing_agent,
    "Underwriting Automation Agent": make_underwriting_automation_agent,
    "Servicing Agent": make_servicing_agent,
    "Regulatory Reporting Agent": make_regulatory_reporting_agent,
    "Compliance Agent": make_compliance_agent,

    # Phase 5
    "KYC Verification Agent": make_kyc_verification_agent,
    "Sanctions Screening Agent": make_sanctions_screening_agent,
    "Adverse News Monitoring Agent": make_adverse_news_monitoring_agent,
    "Transaction Monitoring Agent": make_transaction_monitoring_agent,
    "SAR Preparation Agent": make_sar_preparation_agent,
    "AML Compliance Agent": make_aml_compliance_agent,

    # Phase 6
    "Affordability Assessment Agent": make_affordability_assessment_agent,
    "Complaints Routing Agent": make_complaints_routing_agent,
    "Vulnerability Detection Agent": make_vulnerability_detection_agent,
    "Forbearance Management Agent": make_forbearance_management_agent,

    # Phase 7
    "Product Governance Agent": make_product_governance_agent,
    "Marketing Compliance Agent": make_marketing_compliance_agent,
    "Incentive Monitoring Agent": make_incentive_monitoring_agent,
    "Target Market Monitor Agent": make_target_market_monitor_agent,

    # Phase 8
    "Test Case Generation Agent": make_test_case_generation_agent,
    "Regression Testing Agent": make_regression_testing_agent,
    "UAT Coordination Agent": make_uat_coordination_agent,
    "Compliance Testing Agent": make_compliance_testing_agent,
    "Performance Monitoring Agent": make_performance_monitoring_agent,
    "Readiness Assessment Agent": make_readiness_assessment_agent,

    # Phase 9
    "Application Routing Agent": make_application_routing_agent,
    "Performance Dashboard Agent": make_performance_dashboard_agent,
    "Issue Escalation Agent": make_issue_escalation_agent,
    "Customer Communication Agent": make_customer_communication_agent,
    "Customer Feedback Agent": make_customer_feedback_agent,
    "Feedback Analysis Agent": make_customer_feedback_agent,  # alias

    # Phase 10
    "Portfolio Analytics Agent": make_portfolio_analytics_agent,
    "Default Management Agent": make_default_management_agent,
    "Model Monitoring Agent": make_model_monitoring_agent,
    "Provisioning Agent": make_provisioning_agent,
    "Compliance Monitoring Agent": make_compliance_monitoring_agent,

    # Phase 11
    "Segment Analytics Agent": make_segment_analytics_agent,
    "Market Expansion Agent": make_market_expansion_agent,
    "Portfolio Optimisation Agent": make_portfolio_optimisation_agent,
    "Automation Opportunity Agent": make_automation_opportunity_agent,
    "Risk Scenario Agent": make_risk_scenario_agent,
    "Customer Lifetime Value Agent": make_customer_lifetime_value_agent,
}

SME_PHASE_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "1.1": {
        "name": "Market Analysis and Segmentation",
        "objective": "Define target SME segments, market size and competitive positioning.",
        "waypoint": "Target segments and pain points defined, with indicative market sizing.",
        "primary_agents": [
            "Data Collection Agent",
            "Segmentation Agent",
            "Trend Analysis Agent",
            "Survey Analysis Agent",
        ],
    },
    "1.2": {
        "name": "Business Case Development",
        "objective": "Develop the commercial and capital case for the SME lending product.",
        "waypoint": "Business case approved in principle with clear assumptions and sensitivities.",
        "primary_agents": [
            "Financial Modelling Agent",
            "Competitor Pricing Agent",
            "Cost Analytics Agent",
            "Capital Modelling Agent",
        ],
    },
    "1.3": {
        "name": "Regulatory Environment Assessment",
        "objective": "Clarify regulatory perimeter, permissions and capital requirements conceptually.",
        "waypoint": "Written regulatory classification and indicative capital approach agreed.",
        "primary_agents": [
            "Regulatory Classification Agent",
            "Capital Requirement Calculator Agent",
            "Compliance Checklist Agent",
            "Regulatory Monitoring Agent",
        ],
    },
    "2.1": {
        "name": "Governance and Control Structure",
        "objective": "Define product, credit and data governance and risk appetite.",
        "waypoint": "Governance structure, risk appetite and key policies approved.",
        "primary_agents": [
            "Policy Compliance Agent",
            "Governance Workflow Agent",
            "Risk Monitoring Agent",
            "Data Quality Agent",
        ],
    },
    "2.2": {
        "name": "Credit Risk Framework Development",
        "objective": "Define underwriting, scoring, limits and provisioning approaches conceptually.",
        "waypoint": "Credit framework, decision rules and provisioning approach agreed.",
        "primary_agents": [
            "Financial Analysis Agent",
            "Credit Assessment Agent",
            "Credit Scoring Agent",
            "Decisioning Agent",
            "ECL Calculation Agent",
            "Concentration Monitoring Agent",
        ],
    },
    "3.1": {
        "name": "Product Definition",
        "objective": "Design SME loan products, variants and customer journeys.",
        "waypoint": "Product configuration and journeys documented and endorsed.",
        "primary_agents": [
            "Product Configuration Agent",
            "Customer Journey Agent",
            "Information Provision Agent",
        ],
    },
    "3.2": {
        "name": "Pricing Strategy",
        "objective": "Set risk-based pricing structures aligned to business case and policy.",
        "waypoint": "Pricing grid and policy signed off, consistent with risk appetite.",
        "primary_agents": [
            "Pricing Agent",
            "Cost Analytics Agent",
            "Competitor Pricing Agent",
            "Pricing Analytics Agent",
        ],
    },
    "4.1": {
        "name": "Technology Architecture and Vendors",
        "objective": "Decide how data, decisioning, servicing and reporting will hang together conceptually.",
        "waypoint": "Architecture, integration approach and key vendors agreed.",
        "primary_agents": [
            "Data Integration Agent",
            "Application Processing Agent",
            "Underwriting Automation Agent",
            "Servicing Agent",
            "Regulatory Reporting Agent",
            "Compliance Agent",
        ],
    },
    "5.1": {
        "name": "AML / KYC Framework",
        "objective": "Design financial crime controls across the SME lending lifecycle conceptually.",
        "waypoint": "AML/KYC policy and control set designed at a conceptual level.",
        "primary_agents": [
            "KYC Verification Agent",
            "Sanctions Screening Agent",
            "Adverse News Monitoring Agent",
            "Transaction Monitoring Agent",
            "SAR Preparation Agent",
            "AML Compliance Agent",
        ],
    },
    "6.1": {
        "name": "Lending Standards and Affordability",
        "objective": "Ensure lending is responsible and borrower outcomes are considered conceptually.",
        "waypoint": "Affordability and treatment-of-borrowers expectations defined.",
        "primary_agents": [
            "Affordability Assessment Agent",
            "Information Provision Agent",
            "Complaints Routing Agent",
            "Vulnerability Detection Agent",
            "Forbearance Management Agent",
        ],
    },
    "7.1": {
        "name": "Product Governance and Marketing Conduct",
        "objective": "Align product governance, approvals, marketing and incentives with expectations conceptually.",
        "waypoint": "Target market, governance and marketing checks agreed.",
        "primary_agents": [
            "Product Governance Agent",
            "Marketing Compliance Agent",
            "Incentive Monitoring Agent",
            "Target Market Monitor Agent",
        ],
    },
    "8.1": {
        "name": "Testing and Production Readiness",
        "objective": "Validate that the solution works as intended and controls operate conceptually.",
        "waypoint": "Testing completed with issues addressed to an acceptable level; go-live readiness confirmed.",
        "primary_agents": [
            "Test Case Generation Agent",
            "Regression Testing Agent",
            "UAT Coordination Agent",
            "Compliance Testing Agent",
            "Performance Monitoring Agent",
            "Readiness Assessment Agent",
        ],
    },
    "9.1": {
        "name": "Launch and Initial Operations",
        "objective": "Move from pilot to live operations and stabilise early performance and controls.",
        "waypoint": "Initial portfolio of loans originated with KPIs tracking within expected ranges.",
        "primary_agents": [
            "Application Routing Agent",
            "Performance Dashboard Agent",
            "Issue Escalation Agent",
            "Risk Monitoring Agent",
            "Customer Communication Agent",
            "Feedback Analysis Agent",
        ],
    },
    "10.1": {
        "name": "Ongoing Monitoring and Optimisation",
        "objective": "Monitor portfolio, risk, pricing and compliance, and adapt as needed conceptually.",
        "waypoint": "Stable performance with clear feedback loops into strategy and design.",
        "primary_agents": [
            "Portfolio Analytics Agent",
            "Default Management Agent",
            "Model Monitoring Agent",
            "Provisioning Agent",
            "Pricing Analytics Agent",
            "Compliance Monitoring Agent",
            "Customer Feedback Agent",
        ],
    },
    "11.1": {
        "name": "Strategic Optimisation and Scaling",
        "objective": "Optimise segments, expand markets and increase automation conceptually.",
        "waypoint": "Clear plan for scale-up, with prioritised actions and constraints understood.",
        "primary_agents": [
            "Segment Analytics Agent",
            "Market Expansion Agent",
            "Portfolio Optimisation Agent",
            "Automation Opportunity Agent",
            "Risk Scenario Agent",
            "Customer Lifetime Value Agent",
        ],
    },
}


def describe_phase(phase_id: str) -> str:
    """
    Return a human-readable description of a phase, including objectives and agents.
    """
    if phase_id not in SME_PHASE_DEFINITIONS:
        raise KeyError(f"Unknown phase_id '{phase_id}'.")
    phase = SME_PHASE_DEFINITIONS[phase_id]
    agents = "\n  - " + "\n  - ".join(phase["primary_agents"])
    return (
        f"Phase {phase_id} – {phase['name']}\n\n"
        f"Objective:\n  {phase['objective']}\n\n"
        f"Done when:\n  {phase['waypoint']}\n\n"
        f"Recommended agents:{agents}"
    )


def build_phase_agents(
    phase_id: str,
    mode: ExecutionMode = DEFAULT_MODE,
) -> List[Agent]:
    """
    Instantiate all recommended agents for the given phase.
    """
    if phase_id not in SME_PHASE_DEFINITIONS:
        raise KeyError(f"Unknown phase_id '{phase_id}'.")
    phase = SME_PHASE_DEFINITIONS[phase_id]
    result: List[Agent] = []
    for label in phase["primary_agents"]:
        factory = SME_AGENT_REGISTRY.get(label)
        if factory is None:
            raise KeyError(
                f"No agent factory registered for '{label}'. "
                f"Add it to SME_AGENT_REGISTRY."
            )
        result.append(factory(mode=mode))
    return result


# ============================================================================
# 6. QUICK DEMO / SANITY CHECK
# ============================================================================

print("\n" + "=" * 80)
print("SME BUSINESS LENDING AGENT SYSTEM – READY")
print("=" * 80)
print("Try these in new cells after running this mega-cell:\n")
print("  # 1) Inspect a phase from the framework")
print("  print(describe_phase('2.2'))")
print()
print("  # 2) Build agents for that phase")
print("  phase_22_agents = build_phase_agents('2.2')")
print("  [a.name for a in phase_22_agents]")
print()
print("  # 3) Wrap them in a team and run a mini-workflow (MOCK mode)")
print("  team = AgentTeam('Phase 2.2 – Credit Risk Framework')")
print("  for a in phase_22_agents:")
print("      team.add_agent(a)")
print()
print("  workflow = [")
print("      ('Financial Analysis Agent', 'Analyse this SME case (describe it in the prompt).'),")
print("      ('Credit Assessment Agent', 'Draft a credit assessment building on the prior analysis.'),")
print("      ('Credit Scoring Agent', 'Describe a plausible risk band and score drivers.'),")
print("      ('Decisioning Agent', 'Recommend approve / decline and conditions conceptually.'),")
print("  ]")
print("  results = team.run_sequential_workflow(workflow)")
print()
print("All responses will be MOCKed (no API key required).")
print("The agent structure and phase mapping mirror the SME product framework.")



SME BUSINESS LENDING AGENT SYSTEM – READY
Try these in new cells after running this mega-cell:

  # 1) Inspect a phase from the framework
  print(describe_phase('2.2'))

  # 2) Build agents for that phase
  phase_22_agents = build_phase_agents('2.2')
  [a.name for a in phase_22_agents]

  # 3) Wrap them in a team and run a mini-workflow (MOCK mode)
  team = AgentTeam('Phase 2.2 – Credit Risk Framework')
  for a in phase_22_agents:
      team.add_agent(a)

  workflow = [
      ('Financial Analysis Agent', 'Analyse this SME case (describe it in the prompt).'),
      ('Credit Assessment Agent', 'Draft a credit assessment building on the prior analysis.'),
      ('Credit Scoring Agent', 'Describe a plausible risk band and score drivers.'),
      ('Decisioning Agent', 'Recommend approve / decline and conditions conceptually.'),
  ]
  results = team.run_sequential_workflow(workflow)

All responses will be MOCKed (no API key required).
The agent structure and phase mapping mirror the SME pro

In [5]:
#
#Helper cell: inspect the JSON bundle generated by the build script.
# This cell expects the file: Agentic_Financial_Product_Generation_bundle.json
#to exist in the same directory as this notebook. If it's missing, it will simply print a friendly message.
#

import json
from pathlib import Path

bundle_path = Path("Agentic_Financial_Product_Generation_bundle.json")
if not bundle_path.exists():
    print("Bundle JSON not found. Run the build script first:")
    print("  python build_agentic_artifacts.py")
else:
    bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
    nb = bundle.get("notebook", {})
    mega = bundle.get("mega_cell_code", "")
    print("Bundle summary:")
    print("  - Notebook keys:", list(nb.keys()))
    print("  - Number of cells:", len(nb.get("cells", [])))
    print("  - Length of mega_cell_code (characters):", len(mega))
    if nb.get("cells"):
        first_cell = nb["cells"][0]
        print("  - First cell type:", first_cell.get("cell_type"))
        print("  - First cell source length (lines):", len(first_cell.get("source", [])))

Bundle summary:
  - Notebook keys: ['cells', 'metadata', 'nbformat', 'nbformat_minor']
  - Number of cells: 2
  - Length of mega_cell_code (characters): 61090
  - First cell type: code
  - First cell source length (lines): 1563
